In [ ]:
# Install the required packages:
# - langchain: core LangChain framework with LCEL (LangChain Expression Language) support
# - langchain-openai: OpenAI model integration
!pip install -q langchain langchain-openai


In [ ]:
# IPython utility for displaying images directly in the notebook
from IPython.display import Image

# Colab secret access
from google.colab import userdata

# LangChain message types used in conversations
from langchain.messages import AIMessage, HumanMessage

# Runnable: the base interface for all LCEL components
# RunnablePassthrough: a component that forwards its input unchanged (used for side effects like logging)
from langchain_core.runnables import Runnable, RunnablePassthrough

# The OpenAI chat model wrapper
from langchain_openai import ChatOpenAI
from pathlib import Path
from pydantic import SecretStr

# Securely load the API key
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper function to print a model response with token usage statistics.
# Tracking tokens is important because API costs are based on token consumption.
def print_response(response: AIMessage):
    print(f"Response id: {response.id}")
    if response.usage_metadata is not None:
        # Total input tokens sent to the model in this request
        input_tokens = response.usage_metadata.get("input_tokens", 0)
        # Tokens that were served from cache (cheaper / faster)
        cached_tokens = response.usage_metadata.get("input_token_details", {}).get("cache_read", 0)
        # Total tokens in the model's response
        output_tokens = response.usage_metadata.get("output_tokens", 0)
        # Reasoning tokens are used internally by thinking models (e.g., o1) before producing output
        reasoning_tokens = response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)

        print(f"Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)")

    print()
    print(f"{'-' * 20} [Output] {'-' * 20}")
    print(response.text)

# Helper that renders a LangChain Runnable (model or chain) as a visual graph image.
# LCEL chains expose a .get_graph() method that describes their structure.
def display_graph(runnable: Runnable, output_png: Path) -> None:
    graph = runnable.get_graph()

    # Serialize the graph to a PNG image using Mermaid under the hood
    with output_png.open(mode="wb") as file:
        file.write(graph.draw_mermaid_png())

    # Display the saved image inline in the notebook
    display(Image(output_png, format="png"))


In [ ]:
# Create the OpenAI model with "low" reasoning effort (faster and cheaper; suitable for simple tasks).
openai_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low")

# Build an LCEL chain using the pipe operator (|).
# LCEL (LangChain Expression Language) lets you chain Runnables together like Unix pipes.
# Each component receives the output of the previous component as its input.
#
# Chain breakdown:
#   1. RunnablePassthrough(lambda _: print("Start model execution"))
#      → Runs the lambda as a side effect (logging), then passes the input unchanged to the next step
#   2. openai_model
#      → Sends the conversation to OpenAI and returns an AIMessage response
#   3. RunnablePassthrough(lambda _: print("Model execution finished"))
#      → Logs completion, passes the AIMessage unchanged
#   4. RunnablePassthrough(print_response)
#      → Calls print_response on the AIMessage (prints token stats and content), passes it through
chain = RunnablePassthrough(lambda _: print("Start model execution")) | openai_model | RunnablePassthrough(lambda _: print("Model execution finished")) | RunnablePassthrough(print_response)


In [ ]:
# Build a simple single-message conversation and run it through the chain.
# The chain will log "Start", call the model, log "Finished", then print the response details.
conversation = [HumanMessage("Hello! How are you?")]
result = chain.invoke(conversation)


In [ ]:
# Inspect the `result` variable.
# RunnablePassthrough lets the original value flow through unchanged,
# so `result` is still the AIMessage even after passing through the logging and printing steps.
# NOTE: `result` contains the AIMessage generated by the model. It is preserved due to the use of `RunnablePassthrough`.
result


In [ ]:
# Visualize just the model as a graph.
# A single model has a simple graph: Input → ChatOpenAI → Output.
display_graph(openai_model, Path("/content/model.png"))


In [ ]:
# Visualize the full chain as a graph.
# This shows all four steps: the two passthrough logging steps, the model, and the print step.
# LCEL makes the data flow between components explicit and inspectable.
display_graph(chain, Path("/content/chain.png"))
